# ChakraModel YOLO Cross-Dataset Training (Multi-GPU)

This notebook trains a YOLOv8 object detector across all 5 colonoscopy datasets to ensure perfect generalization for polyp detection. 

**Instructions:**
1. Make sure you have attached your `chakramodel` code dataset to this notebook.
2. Make sure **Internet** is turned ON in the Notebook options on the right panel.
3. Ensure **GPU T4 x2** (or another multi-GPU setup) is selected as the accelerator.
4. Run all cells below.

In [ ]:
!pip install ultralytics requests opencv-python-headless
!pip install gdown

In [ ]:
import os
import cv2
import shutil
import numpy as np
import urllib.request
import zipfile
from pathlib import Path
from ultralytics import YOLO

dataset_dir = Path("/kaggle/working/polyp_dataset")
images_dir = dataset_dir / "images"
labels_dir = dataset_dir / "labels"

# Create YOLO directory structure
for split in ['train', 'val']:
    (images_dir / split).mkdir(parents=True, exist_ok=True)
    (labels_dir / split).mkdir(parents=True, exist_ok=True)

### Step 1: Download & Extract Datasets

In [ ]:
def download_and_extract(url, dest_zip, extract_to):
    if not os.path.exists(dest_zip):
        print(f"Downloading {url}...")
        urllib.request.urlretrieve(url, dest_zip)
    print(f"Extracting {dest_zip}...")
    with zipfile.ZipFile(dest_zip, 'r') as zf:
        zf.extractall(extract_to)

os.makedirs("/kaggle/working/raw_data", exist_ok=True)

# Kvasir-SEG
download_and_extract("https://datasets.simula.no/downloads/kvasir-seg.zip", 
                     "/kaggle/working/kvasir.zip", "/kaggle/working/raw_data/kvasir")

# CVC-ClinicDB (Alternative host since standard requires unrar)
!wget -qO /kaggle/working/clinicdb.zip https://www.dropbox.com/s/p5qv3vwtodcc3va/CVC-ClinicDB.zip?dl=1
with zipfile.ZipFile("/kaggle/working/clinicdb.zip", 'r') as zf:
    zf.extractall("/kaggle/working/raw_data/clinicdb")
    
# CVC-300, ETIS-Larib, CVC-ColonDB (Hosted on PraNet repository / standard links)
!wget -qO /kaggle/working/pranet_datasets.zip https://www.dropbox.com/s/cjpblud1a32qofg/TrainDataset.zip?dl=1
with zipfile.ZipFile("/kaggle/working/pranet_datasets.zip", 'r') as zf:
    zf.extractall("/kaggle/working/raw_data/pranet")


### Step 2: Convert Segmentation Masks to YOLO Bounding Boxes

In [ ]:
def mask_to_yolo(mask_path, img_w, img_h):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bboxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w > 0 and h > 0:
            cx = (x + w/2) / img_w
            cy = (y + h/2) / img_h
            nw = w / img_w
            nh = h / img_h
            bboxes.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
    return bboxes

all_data = []

# 1. Kvasir-SEG
kvasir_imgs = Path("/kaggle/working/raw_data/kvasir/Kvasir-SEG/images")
kvasir_msks = Path("/kaggle/working/raw_data/kvasir/Kvasir-SEG/masks")
if kvasir_imgs.exists():
    for img_p in kvasir_imgs.glob("*.jpg"):
        all_data.append((img_p, kvasir_msks / img_p.name))

# 2. CVC-ClinicDB
clinicdb_imgs = Path("/kaggle/working/raw_data/clinicdb/Original")
clinicdb_msks = Path("/kaggle/working/raw_data/clinicdb/Ground Truth")
if clinicdb_imgs.exists():
    for img_p in clinicdb_imgs.glob("*.png"):
        all_data.append((img_p, clinicdb_msks / img_p.name))
        
# 3. PraNet Datasets (CVC-ColonDB, ETIS, etc. combined in TrainDataset)
pranet_imgs = Path("/kaggle/working/raw_data/pranet/TrainDataset/image")
pranet_msks = Path("/kaggle/working/raw_data/pranet/TrainDataset/mask")
if pranet_imgs.exists():
    for img_p in pranet_imgs.glob("*.png"):
        all_data.append((img_p, pranet_msks / img_p.name))

print(f"Found {len(all_data)} images across all datasets.")

In [ ]:
np.random.seed(42)
np.random.shuffle(all_data)

train_split = int(0.85 * len(all_data))

for i, (img_p, msk_p) in enumerate(all_data):
    split = "train" if i < train_split else "val"
    
    img = cv2.imread(str(img_p))
    if img is None: continue
    h, w = img.shape[:2]
    
    yolo_lines = mask_to_yolo(msk_p, w, h)
    if not yolo_lines: continue
    
    new_img_name = f"{img_p.stem}_{i}.jpg"
    new_txt_name = f"{img_p.stem}_{i}.txt"
    
    cv2.imwrite(str(images_dir / split / new_img_name), img)
    with open(labels_dir / split / new_txt_name, "w") as f:
        f.write("\n".join(yolo_lines))

print("YOLO Dataset generated successfully!")

In [ ]:
yaml_content = f"""path: {dataset_dir}
train: images/train
val: images/val

names:
  0: polyp
"""
with open(dataset_dir / "dataset.yaml", "w") as f:
    f.write(yaml_content)

### Step 3: Train YOLOv8 Model (Multi-GPU)

In [ ]:
model = YOLO("yolov8n.pt") # Using nano for speed, or replace with yolov8x.pt for max accuracy

# Uses device=[0, 1] to train on 2 GPUs in parallel
results = model.train(
    data=str(dataset_dir / "dataset.yaml"),
    epochs=50,
    imgsz=640,
    batch=32, # Increased batch size for 2 GPUs
    device=[0, 1], # Utilizes both GPUs
    project="/kaggle/working/yolo_output",
    name="cross_dataset_polyp"
)

print("Training Complete! Download the weights from /kaggle/working/yolo_output/cross_dataset_polyp/weights/best.pt")